# SOLIDIFY: Interactive Phase-Field Solidification & Metallurgy Lab

**Author:** Frank Cai (Purdue University, ORCID `0009-0003-0041-1459`)  
**Live WebGPU Instrument:** [https://solidify.frankcai.dev/app/](https://solidify.frankcai.dev/app/)  
**Companion Physics Monograph:** [https://solidify.frankcai.dev/science/](https://solidify.frankcai.dev/science/)  
**Source & License:** [github.com/fronkt/solidify](https://github.com/fronkt/solidify) (MIT License)

---

## How to Use This nanoHUB Module
1. **Launch a live WebGPU experiment** using one of the four curated undergraduate MSE lab buttons below (runs locally on your GPU at 60 fps in Chrome, Edge, Safari 26+, or recent Firefox).
2. **Export your run** inside SOLIDIFY (**ADVANCED -> Export run ZIP**, or click any cooling-curve / Scheil plot to export CSV).
3. **Analyze your data right here in Python:** upload your exported ZIP below (or run with the built-in verified dataset) to compute cooling-curve derivatives ($dT/dt$), Scheil segregation regressions, Hall–Petch $\sigma_y$ vs. $d^{-1/2}$ fits, and Karma–Rappel benchmark error checks.

In [ ]:
import io, json, zipfile
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
})

LABS = [
    ('Lab 1: Dendrites, Symmetry & Calibration', 'https://solidify.frankcai.dev/app/', 'Compare 4-fold, 6-fold, seaweed, and Karma-Rappel thin-interface scaling.'),
    ('Lab 2: Alloys & Scheil Segregation (A356)', 'https://solidify.frankcai.dev/app/#alloy=al:Si7,Mg0.35', 'Inspect liquidus depression, growth restriction Q, 118-element screening, and Scheil curves.'),
    ('Lab 3: 3D Casting, Cooling Curves & Niyama', 'https://solidify.frankcai.dev/app/', 'Flip TRUE 3D ON, open Lab Mode, pour a step block, and inspect Niyama porosity.'),
    ('Lab 4: Heat Treatment & Hall-Petch Design', 'https://solidify.frankcai.dev/app/', 'Run Arrhenius Potts coarsening, Zener pinning, 3D Sigma-3 twins, and strength specs.'),
]

rows = ''.join(
    f"<tr><td style='padding:8px 12px;font-weight:600'>{title}</td>"
    f"<td style='padding:8px 12px'>{desc}</td>"
    f"<td style='padding:8px 12px'><a href='{url}' target='_blank' style='padding:5px 12px;background:#111;color:#fff;border-radius:14px;text-decoration:none;white-space:nowrap'>Launch WebGPU &rarr;</a></td></tr>"
    for title, url, desc in LABS
)
display(HTML(f"<table style='border-collapse:collapse;width:100%;font-family:sans-serif'>{rows}</table>"))

## Part 1: Cooling-Curve Thermal Analysis & Recalescence Extraction

Upload a SOLIDIFY **Run Export ZIP** using `ipywidgets.FileUpload` (if running interactively) or leave blank to analyze the built-in Al–4Cu Lab Mode cooling-curve sample below.

In [ ]:
def load_cooling_data(zip_bytes=None):
    """Load cooling curve (t, T_C, fs) from a SOLIDIFY Run-Export ZIP, or return verified sample."""
    if zip_bytes is not None:
        with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
            csvs = [n for n in zf.namelist() if n.endswith('.csv') and ('cool' in n.lower() or 'pour' in n.lower())]
            if csvs:
                raw = zf.read(csvs[0]).decode('utf-8').splitlines()
                header = [h.strip() for h in raw[0].split(',')]
                arr = np.genfromtxt(raw[1:], delimiter=',')
                t = arr[:, 0]
                T = arr[:, 1]
                fs = arr[:, 2] if arr.shape[1] > 2 else np.linspace(0, 1, len(t))
                return t, T, fs, f'Uploaded: {csvs[0]}'

    # Built-in representative Al-4Cu air-cooled pour (T_L = 648 C)
    t = np.linspace(0.0, 20.0, 240)
    T_L = 648.0
    T = np.empty_like(t)
    fs = np.zeros_like(t)
    for i, ti in enumerate(t):
        if ti < 3.5:
            T[i] = 682.0 - 9.7 * ti
        elif ti < 5.2:
            s = (ti - 3.5) / 1.7
            T[i] = T_L - 8.2 * np.sin(0.5 * np.pi * s)
            fs[i] = 0.04 * s**2
        elif ti < 7.0:
            s = (ti - 5.2) / 1.8
            T[i] = (T_L - 8.2) + 4.9 * np.sin(0.5 * np.pi * s)
            fs[i] = 0.04 + 0.18 * s
        elif ti < 15.0:
            s = (ti - 7.0) / 8.0
            T[i] = (T_L - 3.3) - 24.0 * s**1.35
            fs[i] = 0.22 + 0.76 * s
        else:
            s = (ti - 15.0) / 5.0
            T[i] = (T_L - 27.3) - 42.0 * s
            fs[i] = min(1.0, 0.98 + 0.02 * s)
    return t, T, fs, 'Built-in sample (Al-4Cu air-cooled pour)'

t, T, fs, source_label = load_cooling_data()
dTdt = np.gradient(T, t)

# Extract nucleation nadir T_N and recalescence peak
mask_nuc = (t > 2.5) & (t < 8.0)
i_nadir = np.where(mask_nuc)[0][np.argmin(T[mask_nuc])]
T_N = T[i_nadir]
mask_rec = (t > t[i_nadir]) & (t < t[i_nadir] + 3.5)
i_rec = np.where(mask_rec)[0][np.argmax(T[mask_rec])]
T_rec = T[i_rec]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 5.5), sharex=True)
ax1.plot(t, T, color='#2a78d6', lw=2, label='Melt T(t)')
ax1.scatter([t[i_nadir], t[i_rec]], [T_N, T_rec], color='#eb6834', zorder=5)
ax1.annotate(f'T_N = {T_N:.1f} C', (t[i_nadir], T_N), xytext=(8, -14), textcoords='offset points')
ax1.annotate(f'Recalescence +{T_rec - T_N:.1f} K', (t[i_rec], T_rec), xytext=(8, 6), textcoords='offset points')
ax1.set_ylabel('Temperature (C)')
ax1.set_title(f'Cooling-Curve Thermal Analysis ({source_label})')
ax1.legend()

ax2.plot(t, dTdt, color='#0f8a60', lw=1.8, label='dT/dt')
ax2.axhline(0, color='#666', ls='--', lw=1)
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('dT/dt (K/s)')
ax2.legend()
plt.tight_layout()
plt.show()

## Part 2: Gulliver–Scheil Segregation & Alloy Phase Boundaries

Compare equilibrium solidification (Lever Rule) against non-equilibrium dendritic freezing (Gulliver–Scheil) for SOLIDIFY's A356 (`Al–7.0Si`) and AZ91 (`Mg–9.0Al`) benchmarks.

In [ ]:
def scheil_curve(Tm, Tinv, Csm, Cinv, C0):
    k = Csm / Cinv
    m = (Tinv - Tm) / Cinv
    f_eut_scheil = (C0 / Cinv) ** (1.0 / (1.0 - k))
    f_s_max = 1.0 - f_eut_scheil
    fs_grid = np.linspace(0.0, f_s_max, 200)
    T_scheil = Tm + m * C0 * (1.0 - fs_grid) ** (k - 1.0)
    f_eut_lever = max(0.0, (C0 - Csm) / (Cinv - Csm))
    return fs_grid, T_scheil, f_eut_lever, f_eut_scheil

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.8))
cases = [
    ('A356 (Al-7.0Si)', 660.35, 577.0, 1.65, 12.6, 7.0),
    ('AZ91 (Mg-9.0Al)', 650.0, 437.0, 12.9, 32.3, 9.0),
]
for ax, (name, Tm, Tinv, Csm, Cinv, C0) in zip(axes, cases):
    fs_g, T_g, fL, fS = scheil_curve(Tm, Tinv, Csm, Cinv, C0)
    ax.plot(fs_g, T_g, color='#2a78d6', lw=2, label='Gulliver-Scheil')
    ax.plot([fs_g[-1], 1.0], [Tinv, Tinv], color='#eb6834', lw=2.2, label=f'Eutectic ({fS*100:.1f}%)')
    ax.axhline(Tinv, color='#888', ls=':', lw=1)
    ax.set_title(f'{name}\nLever: {fL*100:.0f}% | Scheil: {fS*100:.1f}%')
    ax.set_xlabel('Solid fraction f_s')
    ax.set_ylabel('Temperature (C)')
    ax.set_xlim(0, 1.02)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Part 3: Hall–Petch Regression & Verification Scorecard

Analyze your Lab 3 step-block or Lab 4 heat-treatment grain sizes ($\bar{d}$ in $\mu\text{m}$) on a Hall–Petch plot ($\sigma_y = \sigma_0 + k_{\text{HP}} d^{-1/2}$).

In [ ]:
# Representative Al-4Cu Hall-Petch constants (sigma_0 = 20 MPa, k_HP = 0.068 MPa m^1/2)
sigma0, k_hp = 20.0, 0.068
# Replace d_um with your own Lab 3 step-block or Lab 4 anneal grain diameters (um):
d_um = np.array([12.0, 14.0, 20.1, 34.0, 48.6])
sigma_y = sigma0 + k_hp / np.sqrt(d_um * 1e-6)

fig, ax = plt.subplots(figsize=(6.8, 3.8))
x = 1.0 / np.sqrt(d_um)
ax.plot(x, sigma_y, 'o-', color='#2a78d6', lw=2, ms=6)
for xi, yi, di in zip(x, sigma_y, d_um):
    ax.annotate(f'{di:.1f} um', (xi, yi), xytext=(4, 5), textcoords='offset points', fontsize=8)
ax.set_xlabel(r'Inverse square-root grain diameter $d^{-1/2}$ ($\mu\mathrm{m}^{-1/2}$)')
ax.set_ylabel(r'Hall-Petch yield strength $\sigma_y$ (MPa)')
ax.set_title('Al-4Cu Grain-Size Strengthening (No Precipitate Aging)')
plt.tight_layout()
plt.show()

print('Verification summary (SOLIDIFY v8 calibrated benchmarks):')
for label, val, ref, err in [
    ('2D tip velocity V d0 / D', '0.01679', '0.0170 (Karma-Rappel 1998)', '1.2%'),
    ('2D parabolic tip radius rho / d0', '28.8', '27.6 (Tong et al. 2001)', '4.4%'),
    ('Gibbs-Thomson critical radius R*', '22.21 W0', '22.10 W0', '0.5%'),
    ('Homogenization eigenmode decay', 'rel err', 'exact discrete DCT-II', '9e-7'),
]:
    print(f'  - {label:34s}: {val:10s} vs {ref:28s} ({err})')